In [ ]:

import pandas as pd
import sqlite3
import os
import re

def process_agent_team_list(base_path):
    # Base directory for the project (centralised so it can be easily changed)

    # Bronze layer: raw ingested files (source data)
    source_path = os.path.join(base_path, "Bronze")
    # Silver layer: cleaned and transformed data
    silver_path = os.path.join(base_path, "Silver")

    # --------------------------------------------------------------------------------
    # Identify all input files related to the Agent Team List
    # Assumes naming pattern like: "Agents Team List YYYYMMDD.xlsx"
    all_files = [
        f for f in os.listdir(source_path)
        if f.startswith("Agents Team List") and f.endswith(".xlsx")
    ]

    # Process all files found in the source folder
    for file_name in all_files:

        # Extract date (YYYYMMDD) from file name
        file_date_str = re.search(r"\d{8}", file_name).group()

        print(f"Processing: {file_name}")

        # Build full file path
        file_path = os.path.join(source_path, file_name)

        # Load Excel file into pandas DataFrame
        dim_df = pd.read_excel(file_path)

        # DATA CLEANSING & STANDARDISATION
        # --------------------------------------------------------------------------------
        # Convert date fields to datetime format
        dim_df["date_from"] = pd.to_datetime(dim_df["date_from"], dayfirst=True, errors="coerce")
        dim_df["date_to"] = pd.to_datetime(dim_df["date_to"], dayfirst=True, errors="coerce")

        # Flag current records
        dim_df["flag_current"] = dim_df["date_to"].isna().astype(int)

        # Handle duplicate versions for same employee and start date
        dim_df["version_id"] = dim_df.groupby(["employee_id", "date_from"]).cumcount()

        # --------------------------------------------------------------------------------
        # Convert datetime to string format (YYYY-MM-DD)
        dim_df["date_from"] = dim_df["date_from"].dt.strftime("%Y-%m-%d")
        dim_df["date_to"] = dim_df["date_to"].dt.strftime("%Y-%m-%d")

        # Replace null date_to with far future date
        dim_df["date_to"] = dim_df["date_to"].fillna("2099-12-31")

        ##handle here empty emploee_ids, flag it, and assign some large number to it
        ##later for glod layer it should be removed)
        
        # SQLITE TEMP ENGINE (FOR WINDOW FUNCTIONS)
        # --------------------------------------------------------------------------------
        conn = sqlite3.connect(":memory:")

        # Load DataFrame into SQLite table
        dim_df.to_sql("dim_df", conn, index=False, if_exists="replace")

        query = """
        SELECT
            employee_id,employee_name,date_from,
            CASE
                WHEN next_date_from IS NOT NULL THEN date(next_date_from, '-1 day')
                ELSE date_to
            END as date_to,
            region,manager,flag_current
        FROM (
            SELECT
                *,
                LEAD(date_from) OVER (PARTITION BY employee_id ORDER BY date_from, version_id) as next_date_from
            FROM dim_df
        )
        """

        # Execute SQL transformation
        dim_clean = pd.read_sql(query, conn)

        # DERIVED FIELDS
        # --------------------------------------------------------------------------------
        dim_clean["name_key"] = (dim_clean["employee_name"].astype(str).str.strip().str.upper())

        # ADD SURROGATE KEY
        # --------------------------------------------------------------------------------
        dim_clean = dim_clean.sort_values(["employee_id", "date_from"]).reset_index(drop=True)
        dim_clean["sk_employee"] = dim_clean.index + 1
        dim_clean = dim_clean[["sk_employee"] + [c for c in dim_clean.columns if c != "sk_employee"]]

        # OUTPUT TO SILVER LAYER
        # --------------------------------------------------------------------------------
        output_file = f"dim_clean_{file_date_str}.csv"
        full_path = os.path.join(silver_path, output_file)
        dim_clean.to_csv(full_path, index=False)
        print(f"Saved: {full_path}")




In [ ]:

base_path = r"C:\CallAgents"
process_agent_team_list(base_path)

In [ ]:
import pandas as pd
import os
import numpy as np
from pandasql import sqldf

def process_telephony_pipeline(base_path):

    # --------------------------------------------------------------------------------
    # SQL ENGINE SETUP
    # --------------------------------------------------------------------------------
    # We use pandasql to perform "Non-Equi Joins" (joins using >= or BETWEEN).
    # This is necessary because standard Pandas 'merge' cannot easily join on date ranges.
    pysqldf = lambda q: sqldf(q, globals())

    # --------------------------------------------------------------------------------
    # DIRECTORY CONFIGURATION
    # --------------------------------------------------------------------------------
    source_path = os.path.join(base_path, "Bronze")
    silver_path = os.path.join(base_path, "Silver")

    # --------------------------------------------------------------------------------
    # 1. LOAD LATEST AGENT DIMENSION
    # --------------------------------------------------------------------------------
    # The Dimension table contains the Surrogate Keys (sk_employee).
    # These keys are unique to a specific person during a specific time period.
    dim_files = [f for f in os.listdir(silver_path) if f.startswith("dim_clean") and f.endswith(".csv")]

    if len(dim_files) > 0:
        # Get the newest dimension file based on file system modification time
        latest_dim_file = max(dim_files, key=lambda x: os.path.getmtime(os.path.join(silver_path, x)))
        dim_clean = pd.read_csv(os.path.join(silver_path, latest_dim_file))
        
        # Format dates to strings (YYYY-MM-DD) so the SQL engine can compare them correctly
        dim_clean["date_from"] = pd.to_datetime(dim_clean["date_from"]).dt.strftime('%Y-%m-%d')
        dim_clean["date_to"] = pd.to_datetime(dim_clean["date_to"]).dt.strftime('%Y-%m-%d')
        
        # Create a standardised key for matching (Removes spaces and ignores casing)
        dim_clean["name_key"] = dim_clean["employee_name"].astype(str).str.strip().str.upper()
    else:
        dim_clean = None
        print("CRITICAL: No Dimension file found. Surrogate Keys (sk) cannot be assigned.")

    # --------------------------------------------------------------------------------
    # 2. FILE SELECTION
    # --------------------------------------------------------------------------------
    # Identify all raw Telephony files in the Bronze folder
    all_files = [f for f in os.listdir(source_path) if f.startswith("Telephony") and f.endswith(".csv")]

    # --------------------------------------------------------------------------------
    # 3. PROCESSING ENGINE
    # --------------------------------------------------------------------------------
    for file_name in all_files:
        file_path = os.path.join(source_path, file_name)
        
        # Extract the YYYYMMDD date from the filename using regex-style digit extraction
        digits = "".join([c for c in file_name if c.isdigit()])
        file_date_str = digits[:8]
        
        print(f"Processing: {file_name}")
        df = pd.read_csv(file_path)

        # --- METADATA & CLEANING ---
        df["file_date"] = pd.to_datetime(file_date_str, format="%Y%m%d")
        df["load_timestamp"] = pd.Timestamp.now()
        
        df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
        df["date_sql"] = df["date"].dt.strftime('%Y-%m-%d')
        
        df["flag_dateWrong"] = df["date"].isna().astype(int)
        df["flag_nameWrong"] = df["agent"].isna().astype(int)
        
        df["agent"] = df["agent"].fillna("NONE#").astype(str)
        numeric_cols = df.select_dtypes(include=["number"]).columns
        df[numeric_cols] = df[numeric_cols].fillna(0)

        # --- 4. THE SURROGATE KEY (SK) LOOKUP ---
        if dim_clean is not None:
            df["name_key"] = df["agent"].astype(str).str.strip().str.upper()

            query = """
            SELECT
                t.*,
                dim.employee_id,
                dim.sk_employee
            FROM df t
            LEFT JOIN dim_clean dim ON t.name_key = dim.name_key
                AND t.date_sql BETWEEN dim.date_from AND dim.date_to
            """

            # ONLY FIX (required for pandasql inside function)
            df = sqldf(query, {"df": df, "dim_clean": dim_clean})

        # --- 5. FORMATTING & OUTPUT ---
        df["date"] = pd.to_datetime(df["date"]).dt.strftime('%d/%m/%Y')
        df["file_date"] = pd.to_datetime(df["file_date"]).dt.strftime('%d/%m/%Y')
        
        if "date_sql" in df.columns:
            df = df.drop(columns=["date_sql"])

        output_file = f"Telephony_{file_date_str}.csv"
        output_path = os.path.join(silver_path, output_file)
        df.to_csv(output_path, index=False)
        print(f"Successfully saved with Surrogate Keys: {output_file}")




In [ ]:
# Example execution
base_path = r"C:\CallAgents"
process_telephony_pipeline(base_path)

In [ ]:
import pandas as pd
import os
from datetime import datetime
from pandasql import sqldf

def process_schedules_pipeline(base_path):

    # --------------------------------------------------------------------------------
    # SQL ENGINE SETUP
    # --------------------------------------------------------------------------------
    # We use 'pandasql' to handle complex joins that are difficult in standard Pandas.
    # This allows us to join the Schedule to the Dimension table using date ranges (BETWEEN).
    pysqldf = lambda q: sqldf(q, globals())

    # --------------------------------------------------------------------------------
    # PATH CONFIGURATION
    # --------------------------------------------------------------------------------
    source_path = os.path.join(base_path, "Bronze")  # Raw ingestion layer
    silver_path = os.path.join(base_path, "Silver")  # Cleaned/Transformed layer

    # --------------------------------------------------------------------------------
    # 1. LOAD LATEST DIMENSION TABLE
    # --------------------------------------------------------------------------------
    # We need the Dimension table to retrieve 'sk_employee'.
    # This ensures that a schedule on a specific date is linked to the correct version 
    # of the employee (e.g., if they changed departments or managers).
    dim_files = [f for f in os.listdir(silver_path) if f.startswith("dim_clean") and f.endswith(".csv")]

    if len(dim_files) > 0:
        # Identify the newest dimension file using the OS modification timestamp
        latest_dim_file = max(dim_files, key=lambda x: os.path.getmtime(os.path.join(silver_path, x)))
        dim_clean = pd.read_csv(os.path.join(silver_path, latest_dim_file))
        
        # Standardise dates to YYYY-MM-DD for SQL comparison logic
        dim_clean["date_from"] = pd.to_datetime(dim_clean["date_from"]).dt.strftime('%Y-%m-%d')
        dim_clean["date_to"] = pd.to_datetime(dim_clean["date_to"]).dt.strftime('%Y-%m-%d')
    else:
        dim_clean = None
        print("Warning: No Dimension file found. SK mapping will be skipped.")

    # --------------------------------------------------------------------------------
    # 2. IDENTIFY SCHEDULE FILES
    # --------------------------------------------------------------------------------
    # Identify all Schedule files in the Bronze layer.
    all_files = [f for f in os.listdir(source_path) if f.startswith("Schedules") and f.endswith(".csv")]

    # --------------------------------------------------------------------------------
    # 3. PROCESSING LOOP
    # --------------------------------------------------------------------------------
    for file in all_files:

        file_date_raw = file.replace("Schedules", "").replace(".csv", "").strip()

        print(f"Processing: {file}")
        file_path = os.path.join(source_path, file)
        df = pd.read_csv(file_path)

        # --- 4. DATA TYPES & METADATA ---
        df["file_date"] = file_date_raw
        df["load_timestamp"] = datetime.now()
        
        df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
        df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
        
        df = df.dropna(subset=["start_time", "end_time"]) 

        # --- 5. WORKFORCE CLEAN MODEL (DE-DUPLICATION / NO OVERLAPS) ---
        df = df.sort_values(["employee_id", "scheduled_date", "start_time", "end_time"])
        clean_rows = []

        # Group by employee and day to check chronological order of shifts
        for (emp_id, day), group in df.groupby(["employee_id", "scheduled_date"]):
            group = group.sort_values("start_time").copy()

            for i in range(len(group)):
                current_row = group.iloc[i].copy()
                # If not the first record, check overlap with previous kept row
                if i > 0:
                    prev_row = clean_rows[-1]

                    # Only adjust if same employee and same day
                    if (prev_row["employee_id"] == emp_id and
                        prev_row["scheduled_date"] == day):

                        # Overlap condition
                        if prev_row["end_time"] > current_row["start_time"]:
                    
                            # Trim previous shift end to current start
                            prev_row["end_time"] = current_row["start_time"]

                            # Update last row in clean_rows
                            clean_rows[-1] = prev_row
                clean_rows.append(current_row)
        clean_df = pd.DataFrame(clean_rows)

        # --- 6. ADD SK_EMPLOYEE (POINT-IN-TIME JOIN) ---
        if dim_clean is not None:

            clean_df["date_sql"] = pd.to_datetime(clean_df["scheduled_date"], dayfirst=True).dt.strftime('%Y-%m-%d')

            query = """
            SELECT t.*, dim.sk_employee
            FROM clean_df t
            LEFT JOIN dim_clean dim 
              ON t.employee_id = dim.employee_id
              AND t.date_sql BETWEEN dim.date_from AND dim.date_to
            """

            # FIX ONLY: pass tables explicitly into pandasql context
            clean_df = sqldf(query, {"clean_df": clean_df, "dim_clean": dim_clean})

            if "date_sql" in clean_df.columns:
                clean_df = clean_df.drop(columns=["date_sql"])

        # --- 7. SAVE TO SILVER ---
        output_file = os.path.join(silver_path, f"Schedules_{file_date_raw}.csv")
        clean_df.to_csv(output_file, index=False)
        print(f"Successfully saved cleaned schedules: {output_file}")



In [ ]:

base_path = r"C:\CallAgents"
process_schedules_pipeline(base_path)